#### Import Required functions

In [0]:
from pyspark.sql.functions import current_timestamp, lit, row_number
from pyspark.sql.window import Window



#### READ RAW DATA

In [0]:
df = spark.table("dbx_demo.raw.crop_production")

#### Adding Audit columns 

In [0]:

df_augmented = df.withColumn("create_dt", current_timestamp()) \
                 .withColumn("update_dt", current_timestamp()) \
                 .withColumn("data_source", lit("databricks_demo"))

# ACTION: Displaying augmented data with new columns
print("ACTION: Displaying augmented data with new columns")
display(df_augmented.limit(10))



#### Data Checks

In [0]:
%skip
# ACTION: Identifying rows with missing or zero Area
missing_area = df.filter(df.Area.isNull() | (df.Area == 0))

# ACTION: Identifying rows with missing or zero Production
missing_production = df.filter(df.Production.isNull() | (df.Production == 0))

# ACTION: Identifying duplicate records based on key columns
duplicates = df.groupBy("State_Name", "District_Name", "Crop_Year", "Season", "Crop") \
               .count().filter("count > 1")

# ACTION: Displaying rows with missing Area for cleaning
print("ACTION: Displaying rows with missing Area for cleaning")
display(missing_area)

# ACTION: Displaying rows with missing Production for cleaning
print("ACTION: Displaying rows with missing Production for cleaning")
display(missing_production)

# ACTION: Displaying duplicate records for cleaning
print("ACTION: Displaying duplicate records for cleaning")
display(duplicates)


Notes:Create a table dbx_demo.silver.cleaned_crop_data as per above analysis and handle the identifies anomalies


#### 

In [0]:
# ACTION: Removing rows with missing or zero Area or Production
df_cleaned = df_augmented.filter(
    (df_augmented.Area.isNotNull()) & (df_augmented.Area != 0) &
    (df_augmented.Production.isNotNull()) & (df_augmented.Production != 0)
)

# ACTION: Removing duplicate records based on key columns, keeping the latest update_dt


window_spec = Window.partitionBy("State_Name", "District_Name", "Crop_Year", "Season", "Crop") \
                    .orderBy(df_cleaned.update_dt.desc())

df_deduped = df_cleaned.withColumn("row_num", row_number().over(window_spec)) \
                       .filter("row_num = 1") \
                       .drop("row_num")

# ACTION: Creating table dbx_demo.silver.cleaned_crop_data
df_deduped.write.format("delta").mode("overwrite").saveAsTable("dbx_demo.silver.cleaned_crop_data")

#### Post load checks

In [0]:
# ACTION: Showing count of records in dbx_demo.silver.cleaned_crop_data
record_count = spark.table("dbx_demo.silver.cleaned_crop_data").count()
display({"Record count": record_count})

# ACTION: Displaying 10 random records from dbx_demo.silver.cleaned_crop_data
from pyspark.sql.functions import rand
df_random = spark.table("dbx_demo.silver.cleaned_crop_data").orderBy(rand()).limit(10)
display(df_random)